# Week 14 — Build a Claude-Powered Chatbot & Tool-Using Agent

**Theme:** LLMs & agentic AI

Every previous project trained a model from scratch on a small dataset. This
week is different: **Claude** is already trained on a huge amount of text and
code. Instead of training, we *prompt* it — and then give it **tools**
(Python functions it can decide to call) to build a minimal **agent**.

**What "agentic AI" means, in one sentence:** instead of Claude only replying
with text, we let it request that *we* run a function, look at the result,
and use it to keep working toward the answer — a loop of "think, act,
observe" instead of a single response.

In [ ]:
!pip install -q anthropic

## 0. Set up your API key

Run the cell below and paste your Anthropic API key when prompted. `getpass`
hides what you type and the key is never written into this notebook file.

In [ ]:
import getpass
import os

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

# claude-opus-5 is Anthropic's most capable widely-available model. If you're
# watching API cost across a full classroom, claude-haiku-4-5 is a much
# cheaper, faster alternative that still works fine for this notebook --
# just change MODEL below.
MODEL = "claude-opus-5"

## 1. Your first API call

Every request is the same shape: a model name, a token budget, and a list of
messages. The reply comes back as a list of content blocks — we check each
block's `.type` before reading it.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "In two sentences, what is machine learning?"}],
)

for block in response.content:
    if block.type == "text":
        print(block.text)

## 2. System prompts: giving Claude a persona

A `system` prompt sets standing instructions that apply to the whole
conversation — tone, role, constraints — separately from the user's actual
question.

In [ ]:
def ask(question, system=None, model=MODEL):
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": question}],
    )
    return next(block.text for block in response.content if block.type == "text")

plain = ask("Explain what a neural network is.")
print("--- Default ---")
print(plain)

pirate = ask(
    "Explain what a neural network is.",
    system="You are a very enthusiastic pirate captain who explains everything using nautical metaphors.",
)
print("\n--- Pirate persona ---")
print(pirate)

## 3. A multi-turn conversation

The API itself has no memory between calls — *you* resend the whole
conversation history each time. We wrap that in a small helper class.

In [ ]:
class Chat:
    def __init__(self, system=None, model=MODEL):
        self.system = system
        self.model = model
        self.history = []

    def send(self, user_message):
        self.history.append({"role": "user", "content": user_message})
        response = client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=self.system,
            messages=self.history,
        )
        reply = next(block.text for block in response.content if block.type == "text")
        self.history.append({"role": "assistant", "content": reply})
        return reply

chat = Chat(system="You are a friendly, concise tutor for first-year AI students.")
print(chat.send("My name is Minjun and I'm learning about RNNs."))
print()
print(chat.send("What's my name, and what was I just learning about?"))

## 4. Give Claude tools: a minimal agent

Claude can't run code by itself — but it *can* tell you "please run this
function with these arguments, then tell me the result." We define two
tools, describe them in JSON, and write the loop that executes whichever
tool Claude asks for.

**Tool 1 — calculator.** We deliberately do NOT use Python's `eval()` on the
model's input (that would let it run arbitrary code) — instead we parse the
expression into a syntax tree and only allow basic arithmetic.

In [ ]:
import ast
import operator

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

def safe_calculate(expression):
    """Evaluate a simple arithmetic expression (+ - * / ** parentheses) safely."""
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression!r}")
    tree = ast.parse(expression, mode="eval")
    return _eval(tree.body)

# quick sanity check
print(safe_calculate("(12 + 8) * 3 - 5 ** 2"))

In [ ]:
def convert_units(value, from_unit, to_unit):
    """Convert between a small set of common units."""
    conversions = {
        ("km", "mi"): lambda v: v * 0.621371,
        ("mi", "km"): lambda v: v / 0.621371,
        ("kg", "lb"): lambda v: v * 2.20462,
        ("lb", "kg"): lambda v: v / 2.20462,
        ("celsius", "fahrenheit"): lambda v: v * 9 / 5 + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5 / 9,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key not in conversions:
        raise ValueError(f"Unsupported conversion: {from_unit} -> {to_unit}")
    return conversions[key](value)

print(convert_units(100, "km", "mi"))

Now we describe both functions as **tools** Claude can choose to call —
`input_schema` tells Claude exactly what arguments each tool expects.

In [ ]:
tools = [
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression using + - * / ** and parentheses.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "e.g. '(12 + 8) * 3'"},
            },
            "required": ["expression"],
        },
    },
    {
        "name": "convert_units",
        "description": "Convert a numeric value between units: km<->mi, kg<->lb, celsius<->fahrenheit.",
        "input_schema": {
            "type": "object",
            "properties": {
                "value": {"type": "number"},
                "from_unit": {"type": "string", "enum": ["km", "mi", "kg", "lb", "celsius", "fahrenheit"]},
                "to_unit": {"type": "string", "enum": ["km", "mi", "kg", "lb", "celsius", "fahrenheit"]},
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    },
]

TOOL_FUNCTIONS = {
    "calculator": lambda input: safe_calculate(input["expression"]),
    "convert_units": lambda input: convert_units(input["value"], input["from_unit"], input["to_unit"]),
}

## 5. The agent loop

This is the core pattern behind every tool-using AI agent:

1. Send the conversation (with `tools` attached) to Claude
2. If `stop_reason == "tool_use"`, run every requested tool and send the
   results back as a `user` turn
3. Repeat until Claude replies with plain text (`stop_reason == "end_turn"`)

In [ ]:
def run_agent(user_message, verbose=True):
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return next(block.text for block in response.content if block.type == "text")

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                if verbose:
                    print(f"[agent] calling tool '{block.name}' with input {block.input}")
                try:
                    result = TOOL_FUNCTIONS[block.name](block.input)
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": block.id, "content": str(result),
                    })
                except Exception as e:
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": block.id,
                        "content": f"Error: {e}", "is_error": True,
                    })
        messages.append({"role": "user", "content": tool_results})

In [ ]:
answer = run_agent(
    "A road trip is 342 kilometers. If gas costs $1.85 per liter and the car "
    "uses 1 liter per 15 km, roughly how much will gas cost in total? Also, "
    "how far is that trip in miles?"
)
print("\nFinal answer:\n", answer)

Watch the printed `[agent] calling tool ...` lines above: Claude decided *on
its own*, from the plain-English question, which tools to call, in what
order, and how to combine the results — that decision-making loop is what
makes this an "agent" rather than a single API call.

## Try it yourself

1. **Add a third tool.** Write a `word_count(text)` function or a
   `roman_numeral(number)` converter, describe it in the `tools` list, add it
   to `TOOL_FUNCTIONS`, and ask a question that needs it.
2. **Break it on purpose.** Ask for a conversion the tool doesn't support
   (e.g. `"convert 10 gallons to liters"`) — does the agent handle the tool's
   error gracefully, or does it get stuck?
3. **Compare personas.** Write two very different system prompts (e.g. "a
   strict professor" vs. "a supportive coach") and ask the same question to
   both — how much does phrasing/tone change vs. the actual content?
4. **Cost awareness.** Look up `claude-opus-5` and `claude-haiku-4-5` pricing.
   For a simple task like unit conversion, would you always reach for the most
   capable (and most expensive) model? Why or why not?